# Feature Engineering for Energy Forecasting

This notebook builds leakage-safe, production-ready features from raw time-series data.

Pipeline goals:
1. Apply data quality and missingness policy from EDA.
2. Create calendar, lag, and rolling features using past-only information.
3. Preserve reproducibility through explicit contracts and saved artifacts.
4. Export feature matrix for modeling.

In [ ]:
# Project import setup.
from pathlib import Path
import sys
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_data

In [ ]:
# Load base dataset.
path = project_root / "data" / "raw" / "data.txt"
data = load_data(path).copy()

print(f"Shape: {data.shape}")
print(f"Index: {type(data.index).__name__}")
print(f"Date range: {data.index.min()} -> {data.index.max()}")
data.head()

In [ ]:
# Forecast contract (match EDA decisions).
TARGET_COL = "Global_active_power"   # Update if your target differs
HORIZON_STEPS = 24
PRIMARY_METRIC = "MAE"
GUARDRAIL_METRICS = ["RMSE", "MAPE"]

if TARGET_COL not in data.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not found. Available columns: {list(data.columns)}")

print({
    "target": TARGET_COL,
    "horizon_steps": HORIZON_STEPS,
    "primary_metric": PRIMARY_METRIC,
    "guardrail_metrics": GUARDRAIL_METRICS
})

## 1. Missingness Policy from EDA

Policy:
1. Treat short gaps with local interpolation.
2. Treat long gaps as outage segments and keep an explicit indicator.
3. Never use future values for imputation logic that can leak information.

In [ ]:
# Gap-aware missingness handling for the target column.

def get_gap_lengths(mask: pd.Series) -> pd.Series:
    groups = (mask != mask.shift(fill_value=False)).cumsum()
    lengths = mask.groupby(groups).transform("sum")
    return lengths

fe = data.copy()
missing_mask = fe[TARGET_COL].isna()
gap_len = get_gap_lengths(missing_mask)

SHORT_GAP_MAX = 3
fe["is_outage_gap"] = (missing_mask & (gap_len > SHORT_GAP_MAX)).astype(int)

# Past-only gap fill to avoid future information leakage.
fe[TARGET_COL] = fe[TARGET_COL].ffill(limit=SHORT_GAP_MAX)

print("Missing after short-gap forward fill:", int(fe[TARGET_COL].isna().sum()))
print("Outage rows flagged:", int(fe["is_outage_gap"].sum()))

## 2. Leakage-Safe Feature Construction

All engineered predictors must be available at forecast time.

In [ ]:
# Calendar, lag, and rolling features (past-only).
fe["hour"] = fe.index.hour
fe["day_of_week"] = fe.index.dayofweek
fe["month"] = fe.index.month
fe["is_weekend"] = (fe["day_of_week"] >= 5).astype(int)

for lag in [1, 2, 3, 24, 48, 168]:
    fe[f"{TARGET_COL}_lag_{lag}"] = fe[TARGET_COL].shift(lag)

for window in [3, 6, 12, 24, 48, 168]:
    fe[f"{TARGET_COL}_roll_mean_{window}"] = fe[TARGET_COL].shift(1).rolling(window=window, min_periods=window).mean()
    fe[f"{TARGET_COL}_roll_std_{window}"] = fe[TARGET_COL].shift(1).rolling(window=window, min_periods=window).std()

# Optional cyclical encoding for periodic behavior.
fe["hour_sin"] = np.sin(2 * np.pi * fe["hour"] / 24)
fe["hour_cos"] = np.cos(2 * np.pi * fe["hour"] / 24)
fe["dow_sin"] = np.sin(2 * np.pi * fe["day_of_week"] / 7)
fe["dow_cos"] = np.cos(2 * np.pi * fe["day_of_week"] / 7)

print("Feature matrix shape (pre-dropna):", fe.shape)
fe.head()

In [ ]:
# Leakage register for traceability.
leakage_register = pd.DataFrame([
    {"feature": "hour/day/month/is_weekend", "safe": True, "why": "Known from timestamp at prediction time"},
    {"feature": "lag_*", "safe": True, "why": "Uses historical observed target"},
    {"feature": "rolling_*", "safe": True, "why": "Shifted by 1 before rolling"},
    {"feature": "is_outage_gap", "safe": True, "why": "Derived from known missingness status"},
])

display(leakage_register)

## 3. Model-Ready Dataset and Export

Align with rolling-origin evaluation and persist processed features.

In [ ]:
# Drop rows that are not trainable due to lag/rolling warm-up and unresolved missing target.
required_cols = [c for c in fe.columns if c != TARGET_COL] + [TARGET_COL]
fe_model = fe[required_cols].dropna().copy()

print("Rows before dropna:", len(fe))
print("Rows after dropna:", len(fe_model))
print("Retained %:", round(len(fe_model) / len(fe) * 100, 2))

feature_cols = [c for c in fe_model.columns if c != TARGET_COL]
X = fe_model[feature_cols]
y = fe_model[TARGET_COL]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Recreate rolling-origin split plan on model-ready data.
def build_rolling_splits(df, n_folds=4, train_days=180, valid_days=30):
    rows = []
    end = df.index.max()
    for fold in range(n_folds, 0, -1):
        valid_end = end - pd.Timedelta(days=(fold - 1) * valid_days)
        valid_start = valid_end - pd.Timedelta(days=valid_days) + pd.Timedelta(hours=1)
        train_end = valid_start - pd.Timedelta(hours=1)
        train_start = train_end - pd.Timedelta(days=train_days) + pd.Timedelta(hours=1)

        train_mask = (df.index >= train_start) & (df.index <= train_end)
        valid_mask = (df.index >= valid_start) & (df.index <= valid_end)
        rows.append({
            "fold": n_folds - fold + 1,
            "train_start": train_start,
            "train_end": train_end,
            "valid_start": valid_start,
            "valid_end": valid_end,
            "train_rows": int(train_mask.sum()),
            "valid_rows": int(valid_mask.sum())
        })
    return pd.DataFrame(rows)

split_plan = build_rolling_splits(fe_model, n_folds=4, train_days=180, valid_days=30)
display(split_plan)

In [ ]:
# Persist artifacts for modeling notebook.
out_dir = project_root / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)

feature_parquet_path = out_dir / "features.parquet"
feature_csv_path = out_dir / "features.csv"
split_path = out_dir / "split_plan.csv"
register_path = out_dir / "leakage_register.csv"

try:
    fe_model.to_parquet(feature_parquet_path)
    feature_saved = feature_parquet_path
except Exception:
    fe_model.to_csv(feature_csv_path)
    feature_saved = feature_csv_path

split_plan.to_csv(split_path, index=False)
leakage_register.to_csv(register_path, index=False)

print("Saved:")
print(feature_saved)
print(split_path)
print(register_path)